# Lambert Liu Runner

In [1]:
import utils as ut
import b_run_staging as b
import h_ll_runner as h
from i_hyper_tuning import Tuner
import c_clustering as c
import numpy as np
import polars as pl
from numba import set_num_threads, get_num_threads

set_num_threads(15)
get_num_threads()

15

### Loading in required data and changing to named tuples

In [2]:
# Loading in required numpy arrays
static_configs = ut.load_json5('static_configs')
runtime_configs = ut.load_json5('runtime_configs')
base_config = ut.merge_configs(static_configs, runtime_configs)

train_test_dict = ut.load_json5("train_test_dict")
bin_metric_dict = ut.load_json5("bin_metric_dict")

user_counts = ut.load_data("user_counts", "df")
user_interactions = ut.load_data("user_interactions", "df")
user_mapping = ut.load_data("user_mapping", "df")

degen_mask = ut.load_data("degen_mask", "np")
interpolation_weights = ut.load_data("interpolation_weights", "np")

# Loading initial grids
u_init = ut.load_data("u_init", "np")
v_init = ut.load_data("v_init", "np")

p_init = ut.load_data("p_init", "np")
u_pos_init = ut.load_data("u_pos_init", "np")
v_pos_init = ut.load_data("v_pos_init", "np")

n_counts_init = ut.load_data("n_counts_init", "np")

u_clustering = ut.load_data("u_clustering", "np")
v_clustering = ut.load_data("v_clustering", "np")

u_pos_clustering = ut.load_data("u_pos_clustering", "np")
v_pos_clustering = ut.load_data("v_pos_clustering", "np")
p_pos_clustering = ut.load_data("p_pos_clustering", "np")


### Converting to named tuples

In [3]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = b.df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = b.df_to_nt('user_counts_nt', user_counts,)
output_idx_nt, model_idx_nt = (b.get_model_and_output_idx_nt())
train_test_nt_class = b.dictionary_to_named_tuple_class('train_test_nt',train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)
bin_metric_nt = b.dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating tuner class for runs

In [4]:
# Creating user type groups for tuning
user_type_groups = (user_mapping.sort('user_id')['source_user_type'].to_numpy() == 'machine').astype(np.int8)

t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, 
          n_counts_init, user_counts_nt, user_interactions_nt, interpolation_weights, bin_metric_nt, output_idx_nt, model_idx_nt, train_test_nt_class, user_type_groups)

# Validation Runs

#### Lambert Liu Unsmoothed runner

We do two things here
- Tests hurdle vs nb model
- Tunes best ll model

In [5]:
hyperparams = ut.load_json5('hyper_choices')

experiment_name = 'model_selection'
for hurdle_model in (True, False):
    results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model,  hyperparams=hyperparams, 
                    train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/9 in 60.4s
finished_config 2/9 in 46.4s
finished_config 3/9 in 48.0s
finished_config 4/9 in 49.7s
finished_config 5/9 in 51.3s
finished_config 6/9 in 52.3s
finished_config 7/9 in 52.4s
finished_config 8/9 in 52.3s
finished_config 9/9 in 52.9s
finished_config 1/9 in 57.9s
finished_config 2/9 in 46.6s
finished_config 3/9 in 46.6s
finished_config 4/9 in 46.6s
finished_config 5/9 in 46.3s
finished_config 6/9 in 46.3s
finished_config 7/9 in 46.7s
finished_config 8/9 in 46.5s
finished_config 9/9 in 47.1s


#### Cluster Smoothing Runner

In [5]:
hurdle_nb_model = ut.load_json5('hurdle_nb_model')
hyperparams = ut.load_json5('hyper_choices')

if hurdle_nb_model['hurdle_model'] is None:
    raise ValueError('Should only be run after selecting hurdle or NB model')

In [ ]:
experiment_name = 'cluster_smoothing'

cluster_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_nb_model['hurdle_model'], 
    hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/100 in 79.7s
finished_config 2/100 in 82.7s
finished_config 3/100 in 69.6s
finished_config 4/100 in 69.8s
finished_config 5/100 in 70.0s
finished_config 6/100 in 70.8s
finished_config 7/100 in 68.9s
finished_config 8/100 in 68.1s
finished_config 9/100 in 68.6s
finished_config 10/100 in 66.8s
finished_config 11/100 in 67.5s
finished_config 12/100 in 67.9s
finished_config 13/100 in 68.4s
finished_config 14/100 in 66.0s
finished_config 15/100 in 67.0s
finished_config 16/100 in 65.7s
finished_config 17/100 in 65.0s
finished_config 18/100 in 64.9s
finished_config 19/100 in 64.6s
finished_config 20/100 in 64.7s
finished_config 21/100 in 67.9s
finished_config 22/100 in 66.9s
finished_config 23/100 in 66.7s
finished_config 24/100 in 66.8s
finished_config 25/100 in 66.5s
finished_config 26/100 in 66.7s
finished_config 27/100 in 66.7s


#### Global Smoothing Runner

In [5]:
hurdle_nb_model = ut.load_json5('hurdle_nb_model')
hyperparams = ut.load_json5('hyper_choices')

if hurdle_nb_model['hurdle_model'] is None:
    raise ValueError('Should only be run after selecting hurdle or NB model')

In [6]:
experiment_name = 'global_smoothing'

global_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_nb_model['hurdle_model'], 
    hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/100 in 94.8s
finished_config 2/100 in 51.0s
finished_config 3/100 in 64.9s
finished_config 4/100 in 53.1s
finished_config 5/100 in 53.0s
finished_config 6/100 in 54.4s
finished_config 7/100 in 53.4s
finished_config 8/100 in 54.2s
finished_config 9/100 in 53.3s
finished_config 10/100 in 53.3s
finished_config 11/100 in 54.0s
finished_config 12/100 in 53.3s
finished_config 13/100 in 53.2s
finished_config 14/100 in 53.4s
finished_config 15/100 in 54.6s
finished_config 16/100 in 53.4s
finished_config 17/100 in 53.4s
finished_config 18/100 in 53.1s
finished_config 19/100 in 53.4s
finished_config 20/100 in 55.4s
finished_config 21/100 in 54.8s
finished_config 22/100 in 54.9s
finished_config 23/100 in 55.4s
finished_config 24/100 in 58.0s
finished_config 25/100 in 61.3s
finished_config 26/100 in 66.6s
finished_config 27/100 in 66.4s
finished_config 28/100 in 66.7s
finished_config 29/100 in 67.9s
finished_config 30/100 in 67.1s
finished_config 31/100 in 66.2s
finished_config 3

# Test final run

#### Lambert Liu Unsmoothed runner

In [ ]:
experiment_name = 'no_smoothing'

nll_only = False
results_type = 'nll_only' if nll_only else 'full'
best_models = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

# Running experiment and storing results
test_results = t.test_run(experiment_name=experiment_name, hurdle_nb_model=hurdle_nb_model, 
                    selected_config=best_models[experiment_name], train_test_dict=train_test_dict, base_config=base_config, 
                    degen_mask=degen_mask, bin_metric_dict=bin_metric_dict, nll_only=nll_only)

ut.store_run_results(results=test_results, dir=f'test/{experiment_name}/{results_type}', run_name=f'{experiment_name}_test')

#### Cluster Smoothing Runner

In [ ]:
experiment_name = 'cluster_smoothing'

nll_only = False
results_type = 'nll_only' if nll_only else 'full'
best_models = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

# Running experiment and storing results
test_results = t.test_run(experiment_name=experiment_name, hurdle_nb_model=hurdle_nb_model, 
                    selected_config=best_models[experiment_name], train_test_dict=train_test_dict, base_config=base_config, 
                    degen_mask=degen_mask, bin_metric_dict=bin_metric_dict, nll_only=nll_only)

ut.store_run_results(results=test_results, dir=f'test/{experiment_name}/{results_type}', run_name=f'{experiment_name}_test')

#### Global smoothing runner

In [ ]:
experiment_name = 'global_smoothing'

nll_only = False
results_type = 'nll_only' if nll_only else 'full'
best_models = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

# Running experiment and storing results
test_results = t.test_run(experiment_name=experiment_name, hurdle_nb_model=hurdle_nb_model, 
                    selected_config=best_models[experiment_name], train_test_dict=train_test_dict, base_config=base_config, 
                    degen_mask=degen_mask, bin_metric_dict=bin_metric_dict, nll_only=nll_only)

ut.store_run_results(results=test_results, dir=f'test/{experiment_name}/{results_type}', run_name=f'{experiment_name}_test')